In [53]:
import pandas as pd
import json
import random

In [54]:
new_triples_dmd = pd.read_csv('../new_triples_dmd_node_exist.csv')
print(new_triples_dmd.shape[0])
new_triples_dmd.head()

945


,relation,x_type,y_type,x_id,y_id
0,disease_phenotype_positive,disease,effect/phenotype,10679,3693
1,disease_phenotype_positive,disease,effect/phenotype,10383,717
2,disease_phenotype_positive,disease,effect/phenotype,10679,3693
3,disease_phenotype_positive,disease,effect/phenotype,10679,5162
4,disease_phenotype_positive,disease,effect/phenotype,10679,6532


In [55]:
nodes = pd.read_csv('../../../kg/nodes.csv')
print(nodes.shape[0])
nodes.head()

151321


,node_index,node_id,node_type,node_name,node_source
0,0,381,gene/protein,ARF5,NCBI
1,1,4074,gene/protein,M6PR,NCBI
2,2,2288,gene/protein,FKBP4,NCBI
3,3,56603,gene/protein,CYP26B1,NCBI
4,4,55471,gene/protein,NDUFAF7,NCBI


In [56]:
nodes_d = nodes.drop_duplicates(['node_id', 'node_type'], keep='first')
nodes_d.shape[0]

151112

In [57]:
df = pd.merge(new_triples_dmd, nodes_d, left_on=['x_id', 'x_type'], right_on=['node_id', 'node_type'], how='left').rename(columns={'node_index': 'x_index'}).get(
    ['relation', 'x_index', 'y_id', 'y_type']
).astype({'x_index': int}).astype({'x_index': str}).astype({'y_id': int}).astype({'y_id': str})

print(df.shape[0])

df = pd.merge(df, nodes_d, left_on=['y_id', 'y_type'], right_on=['node_id', 'node_type'], how='left').rename(columns={'node_index': 'y_index'}).get(
    ['relation', 'x_index', 'y_index']
).astype({'y_index': int}).astype({'y_index': str})

print(df.shape[0])
df.head()

945
945


,relation,x_index,y_index
0,disease_phenotype_positive,27017,30068
1,disease_phenotype_positive,40091,29803
2,disease_phenotype_positive,27017,30068
3,disease_phenotype_positive,27017,32886
4,disease_phenotype_positive,27017,31232


In [58]:
edges = pd.read_csv('../../../kg/edges.csv')
print(edges.shape[0])
edges.head()

14641692


,relation,display_relation,x_index,y_index
0,protein_protein,ppi,0,1898
1,protein_protein,ppi,0,769
2,protein_protein,ppi,0,15031
3,protein_protein,ppi,0,2385
4,protein_protein,ppi,0,4981


In [59]:
edges = pd.concat([
    edges,
    df
])
print(edges.shape[0])
edges.tail()

14642637


,relation,display_relation,x_index,y_index
940,disease_protein,NaN,27513,13241
941,disease_disease,NaN,26788,61384
942,phenotype_phenotype,NaN,37499,35969
943,bioprocess_protein,NaN,74007,8206
944,disease_protein,NaN,27017,9662


In [60]:
with open('entity2id.txt', 'w') as f:
    f.write(f'{len(nodes)}\n')
    for _, row in nodes.iterrows():
        f.write(f'{row["node_name"]}:{row["node_type"]}\t{row["node_index"]}\n')

In [61]:
relations = edges.drop_duplicates(subset=['relation'])['relation'].to_list()
relation2id = {relation: i for i, relation in enumerate(relations)}

with open('relation2id.txt', 'w') as f:
    f.write(f'{len(relations)}\n')
    for k, v in relation2id.items():
        f.write(f'{k}\t{v}\n')

In [62]:
edges['rela_id'] = edges['relation'].apply(lambda x: relation2id[x])
edges = edges[['x_index', 'y_index', 'rela_id']]
data = list(edges.itertuples(index=False, name=None))
random.shuffle(data)
data[:10]

[(146526, 14717, 28),
 (23573, 26013, 5),
 (146549, 8491, 28),
 (49621, 39069, 9),
 (89762, 146419, 28),
 (11692, 11385, 0),
 (2506, 146546, 28),
 (135667, 146319, 28),
 (22947, 23958, 5),
 (3715, 146491, 28)]

In [63]:
train_size = int(len(data) * 0.9)
valid_size = int(len(data) * 0.05)
test_size = len(data) - train_size - valid_size

print('train size: ', train_size)
print('valid size: ', valid_size)
print('test size: ', test_size)

train_data = data[:train_size]
valid_data = data[train_size:train_size+valid_size]
test_data = data[train_size+valid_size:]

train size:  13178373
valid size:  732131
test size:  732133


In [65]:
with open('train2id.txt', 'w') as f:
    f.write(f'{len(train_data)}\n')
    for x, y, r in train_data:
        f.write(f'{x} {y} {r}\n')

with open('valid2id.txt', 'w') as f:
    f.write(f'{len(valid_data)}\n')
    for x, y, r in valid_data:
        f.write(f'{x} {y} {r}\n')

with open('test2id.txt', 'w') as f:
    f.write(f'{len(test_data)}\n')
    for x, y, r in test_data:
        f.write(f'{x} {y} {r}\n')